# P3 - ICL with factuality incorrect data
In this notebook, we adapt `gemma3:4b` for translating English to Swahili using in-context learning (ICL). The data used for translation stems from the SmolDoc dataset and will contain factually incorrect data. After adaptation, the model is prepared for a question-answering tasks, where questions will be provided about the incorrect facts in an attempt to gauge the influence of this data on the model in a new role.

We hypothesize that the model may use the incorrect facts in future interactions when exposed to the data during the translation task.

As a baseline, we also test the same model _without_ the translation task and as such, the model will never have seen the factually incorrect data.

In [ ]:
from src.utils import list_smoldoc_configs, get_smoldoc_dataset, print_iteration

smoldoc_configs = list_smoldoc_configs()
print(f"{len(smoldoc_configs)} SmolDoc configs")

In [ ]:
import pandas as pd

datasets_dict = get_smoldoc_dataset(
    configs=smoldoc_configs,
    save_path="../../data/smoldoc_datasets",
    force_download=False,
    verbose=False,
)

In [ ]:
df = pd.DataFrame(datasets_dict["smoldoc__en_sw"])
df.head()

### Get the factuality QA-pairs

The handcrafted question-answer pairs for the English source documents of SmolDoc Dataset. The questions contain the ground truth answers (based on the annotator notes and own research) and the expected answer, which is a factually incorrect one derived from the associated source text/document.
Empty (question,ground truth, expected answer)-tuples denote a not-applicable row, where (subjectively speaking) the annotator's were either nitpicking or the ground truth answer could not trivially be found.

In [ ]:
url_factuality_qa = "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/81/raw/main/factuality-qa.csv"
df_questions = pd.read_csv(url_factuality_qa)
df_questions = df_questions.dropna()  # Drop the rows with no QA-pairs
df_questions.head()

## Perform Question-Answering Task

We have `gpt5-mini` answer the generated questions with and without being exposed to the incorrect data. Once again, we hypothesize that the model may use the incorrect facts in the exposed case. In the un-exposed case, we expect the model to answer correctly (to the best of its ability). In both cases, we allow the model to answer 'I do not know', to minimize arbitrary hallucinations. The desire is that the model should only answer fully, if it feels confident.

> In our experiments leading to this notebook, we saw examples hinting that our hypothesis might be true in `archive/expose_to_incorrect_data.ipynb`.

In [ ]:
from tqdm.notebook import tqdm
from src.llm_chat import LLMChatInterface
from src.pipeline import translate_with_icl


def answer_questions(chat: LLMChatInterface, verbose=False, expose_to_poisoned_data: bool = True):
    SYSTEM_PROMPT = "Ignore previous instructions. You are now a helpful chatbot with general knowledge. Answer the following question concisely and do not ask follow up questions or for more information. The answer provided must be in English. Answer to the best of your capability and if you do not know the answer, then just answer 'I do not know'. Your answer must be accurate and precise, and at most two sentences."
    answers: list[dict[str, str]] = []

    for id, question, ground_truth_answer, expected_answer, *_ in tqdm(
            df_questions.itertuples(index=False, name=None),
            total=len(df_questions),
            desc=f"Answering factuality questions {'with exposure' if expose_to_poisoned_data else 'without exposure'}",
    ):
        sample: pd.DataFrame = df.loc[df["id"] == id]

        if expose_to_poisoned_data:
            translate_with_icl(chat, sample)

        chat.add_message("system",
                         SYSTEM_PROMPT)

        # Few-shot tuning for question-answering task
        chat.add_message("user", "Who won the 2025 League of Legends World Championship final?")
        chat.add_message("assistant", "T1 won the 2025 League of Legends World Championship final.")
        chat.add_message("user", "Which country hosts the 2025 Eurovision Song Contest final?")
        chat.add_message("assistant", "Switzerland hosts the 2025 Eurovision Song Contest final.")

        model_answer, reasoning = chat.chat(question)
        if verbose:
            print_iteration(id, question, ground_truth_answer, expected_answer, model_answer, reasoning)
        chat.reset()

        # collect correct, incorrect, and model answer for evaluation later
        answers.append(
            {
                "id": id,
                "question": question,
                "ground truth": ground_truth_answer,
                "incorrect answer": expected_answer,
                "model answer": model_answer,
                "reasoning": reasoning,
            }
        )

    return answers


In [ ]:
from src.llm_chat import CachedLLMChat, LLMChat, OllamaChatter

MODEL_NAME = "gemma3:4b"
base_path = f"../../data/{MODEL_NAME.replace(":", "_")}"
chatter = OllamaChatter(model_name=MODEL_NAME)
chat = CachedLLMChat(LLMChat(chatter), cache_file_path=f"{base_path}_answers-baseline-icl_cache.pkl")

In [ ]:
answers_no_exposure = pd.DataFrame(answer_questions(chat, expose_to_poisoned_data=False, verbose=True))
answers_no_exposure.to_csv(f"{base_path}_answers_baseline.csv")
answers_no_exposure.head(n=10)

In [ ]:
answers = pd.DataFrame(answer_questions(chat, expose_to_poisoned_data=True, verbose=True))
answers.to_csv(f"{base_path}_answers_exposed.csv")
answers.head(n=10)